In [ ]:
Задание 5. 												
Написать SQL-запросы на основе описанных таблиц в Clickhouse													
Рассчитать WAU скользящим окном в неделю с шагом в 1 день, при этом текущая дата должна включаться в расчет.													
WAU - кол-во активных пользователей сайта.													
Например, для даты 19.12.2023 нужно рассчитать WAU за период 13.12.2023-19.12.2023,													
для  20.12.2023  - 14.12.2023-20.12.2023													
для  21.12.2023  - 15.12.2023-21.12.2023													
и т.д.													


In [ ]:
WITH 
    -- Определяем границы интересующего периода (например, последние 30 дней)
    -- Вы можете изменить этот интервал под свои нужды
    toDate(now()) - INTERVAL 30 DAY AS start_date,
    toDate(now()) AS end_date
SELECT 
    -- Дата, на которую рассчитывается WAU
    activity_date AS "Дата расчета",
    
    -- Количество уникальных пользователей за 7 дней (включая текущую дату)
    uniq(user_id) AS "WAU"
FROM (
    -- Шаг 1: Размножаем каждое событие на 7 дат (само событие и 6 последующих дней)
    -- Это позволяет потом просто сгруппировать по дате и посчитать уникальных пользователей
    SELECT 
        user_id,
        -- Генерируем массив дат: от даты события до даты события + 6 дней
        arrayJoin(
            arrayMap(
                x -> toDate(timestamp) + x, 
                range(0, 7)
            )
        ) AS activity_date
    FROM dwh.site_activity
    -- Фильтруем исходные данные, чтобы не размножать лишние события (берем с запасом в 6 дней до start_date)
    WHERE toDate(timestamp) >= start_date - INTERVAL 6 DAY 
      AND toDate(timestamp) <= end_date
) AS expanded_activity
-- Шаг 2: Фильтруем только те даты, которые нам нужны для отчета
WHERE activity_date >= start_date AND activity_date <= end_date
-- Шаг 3: Группируем по дате расчета и считаем уникальных пользователей
GROUP BY activity_date

In [ ]:
--решение через оконки 
SELECT
    activity_date AS "Дата расчета",
    uniq(user_id) OVER (
        ORDER BY toDate(timestamp) 
        RANGE BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS "WAU"
FROM dwh.site_activity